In [2]:
from langchain_openai.chat_models import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.types import Send
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import DocumentStream

import math
import operator
from typing import TypedDict, List, Optional, Annotated, Literal, Set
from pydantic import BaseModel
import requests
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
)
import os

from readability import Document
from markdownify import markdownify as md

import spacy
nlp = spacy.load("en_core_web_sm")

from dotenv import load_dotenv
load_dotenv('../.env')
EMAIL_ADDRESS = os.environ.get('EMAIL_ADDRESS')

/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# The PDF file containing the research paper
#from docling.document_converter import DocumentConverter

#converter = DocumentConverter()
#result = converter.convert("../data/simucell3d-nat-comp-sci-paper.pdf")

#markdown = result.document.export_to_markdown()

#with open("converted_pdf.md", "r") as file:
#    markdown = file.read()

#print(markdown)

In [3]:
import json
from pathlib import Path

json_path = Path('/Users/srunser/Documents/CitationChecker/frontend/my-app/src/data/simucell3d-nat-comp-sci-paper-segmented-tokens.json')

with json_path.open('r', encoding='utf-8') as f:
    segmented_text_tokens = json.load(f)

print(f'Loaded {len(segmented_text_tokens)} segmented tokens')
segmented_text_tokens[:3]

# Extract all sentences from the segmented text tokens and group them by sentence_id
sentences = {token["sentenceId"] : token["sentence"] for token in segmented_text_tokens}

full_text = "\n".join(sentences[sentence_id].replace("\n", " ") for sentence_id in sorted(sentences.keys()))
print(f'Full text length: {len(full_text)} characters')


Loaded 5152 segmented tokens
Full text length: 65613 characters


# References Extraction

In [4]:
# Create the model
llm_low_temp = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [ ]:
import json
from langgraph.config import get_stream_writer


def reference_extraction_node(state: ReferenceExtractionState) -> ReferenceExtractionState:
    """
    Extract references from the bibliography section.
    """

    document = state.get("document", "")
    writer = get_stream_writer()

    system_prompt = """
You are an expert at analyzing references in scientific articles.
Find all references in the bibliography section.

Output format (strict):
- Output one JSON object per line (JSONL)
- No markdown
- No commentary
- No surrounding array

Each JSON object must follow:
{
  "ref_id": int,
  "journal": str,
  "title": str,
  "authors": [str] | null,
  "publication_year": int | null,
  "doi": str | null
}
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=document),
    ]

    partial_references: List[Reference] = []
    seen_keys = set()
    buffer = ""

    def emit_reference_line(line: str) -> None:
        """Extract the reference from the line and emit in real time"""
        line = line.strip().rstrip(",")
        if not line:
            return

        try:
            item = json.loads(line)
            ref = Reference.model_validate(item)
        except Exception:
            return

        key = (ref.ref_id, (ref.title or "").strip().lower())
        if key in seen_keys:
            return

        seen_keys.add(key)
        partial_references.append(ref)
        writer(
            {
                "event": "reference_extracted",
                "count": len(partial_references),
                "ref_id": ref.ref_id,
                "ref": ref.model_copy(),
            }
        )


    # Stream the raw output of the llm, parse each line, convert it 
    # in a reference object and emit it for further processing
    for chunk in llm_low_temp.stream(messages):
        content = chunk.content
        if not content:
            continue
        if isinstance(content, list):
            content = "".join(str(x) for x in content)

        buffer += content

        while "\n" in buffer:
            line, buffer = buffer.split("\n", 1)
            emit_reference_line(line)

    emit_reference_line(buffer)
    return {"references": partial_references}


builder = StateGraph(ReferenceExtractionState)
builder.add_node("reference_extraction_node", reference_extraction_node)
builder.add_edge(START, "reference_extraction_node")
builder.add_edge("reference_extraction_node", END)
graph = builder.compile()

initial_state: ReferenceExtractionState = {
    "document": full_text,
    "references": [],
}

final_state = None

# Stream the reference extraction in real time
for mode, data in graph.stream(initial_state, stream_mode=["custom", "updates"]):
    if mode == "custom" and data.get("event") == "reference_extracted":
        print(f"[{data['count']}] id={data['ref_id']} | {data['ref'].title}")
    elif mode == "updates":
        final_state = data

# Now final_state contains the complete output
references = final_state.get('reference_extraction_node', {}).get('references', [])
print(f"Total references extracted: {len(references)}")

[1] id=1 | An equatorial contractile mechanism drives cell elongation but not cell division.
[2] id=2 | Collective cell migration in morphogenesis, regeneration and cancer.
[3] id=3 | The extracellular matrix in development.
[4] id=4 | Forces in tissue morphogenesis and patterning.
[5] id=5 | Measuring mechanical stress in living tissues.
[6] id=6 | Measuring forces and stresses in situ in living tissues.
[7] id=7 | Microscale interrogation of 3D tissue mechanics.
[8] id=8 | The mechanical properties of the cell surface: III. The sea-urchin egg from fertilization to cleavage.
[9] id=9 | From molecules to cells: imaging soft samples with the atomic force microscope.
[10] id=10 | The optical stretcher: a novel laser tool to micromanipulate cells.
[11] id=11 | Mechanisms of pulsed laser ablation of biological tissues.
[12] id=12 | A mathematical model for outgrowth and spatial patterning of the vertebrate limb bud.
[13] id=13 | Video force microscopy reveals the mechanics of ventral furro

In [16]:
def fetch_crossref_metadata(ref: Reference) -> Reference:
    """
        Get the following metadata about the paper from crossref:
        - author names
        - journal name
        - html url
        - doi
    """
    crossref_request_params = {
                "query.title": ref.title,
                "rows": 5,
    }
    if ref.publication_year is not None:
        crossref_request_params["filter"] = (
            f"from-pub-date:{ref.publication_year}-01-01,"
            f"until-pub-date:{ref.publication_year}-12-31"
        )

    crossref_response = requests.get(
        "https://api.crossref.org/works",
        params=crossref_request_params,
        timeout=20,
    ).json()

    if crossref_response.get("message") and crossref_response["message"].get("items"):
        crossref_item = crossref_response["message"]["items"][0]

        # Get the URL to the paper
        ref.html_url = crossref_item.get("URL") or ref.html_url

        # Get its DOI to properly identify it
        ref.doi = crossref_item.get("DOI") or ref.doi

        # Get the journal name
        ref.journal = (
            (crossref_item.get("container-title") or [ref.journal])[0]
            if isinstance(crossref_item.get("container-title"), list)
            else crossref_item.get("container-title") or ref.journal
        )

        # Get the full author list, if available
        authors = []
        for author in crossref_item.get("author", []):
            given = author.get("given", "").strip()
            family = author.get("family", "").strip()
            full_name = f"{given} {family}".strip()
            if full_name:
                authors.append(full_name)
        if authors:
            ref.authors = authors
    return ref

def fetch_openaccess_metadata(ref: Reference) -> Reference:
    """
    Get the following metadata from unpaywall API:
    - is_openaccess
    - url_pdf (A URL directly pointing to the pdf download endpoint)
    """
    if not ref.doi:
        return ref

    unpaywall_url = f"https://api.unpaywall.org/v2/{ref.doi}?email={EMAIL_ADDRESS}"
    unpaywall_response = requests.get(unpaywall_url, timeout=20).json()

    ref.is_open_access = bool(unpaywall_response.get("is_oa", False))

    # Use OA location if present
    best_oa_location = unpaywall_response.get("best_oa_location") or {}
    if ref.is_open_access:
        ref.pdf_url = best_oa_location.get("url_for_pdf") or ref.pdf_url
        ref.html_url = best_oa_location.get("url_for_landing_page") or ref.html_url

    return ref

def fetch_paper_content(ref: Reference) -> Reference:
    """Load the content of the paper and save it as markdown file"""

    # Only load the content of openaccess articles
    if ref.is_open_access:

        # If we have access to the pdf download endpoint
        if ref.pdf_url:
            pdf_request_response = requests.get(ref.pdf_url, timeout=30)
            if pdf_request_response.status_code == 200:
                pdf_stream = BytesIO(pdf_request_response.content)
                converter = DocumentConverter()
                result = converter.convert(DocumentStream(name="paper.pdf", stream=pdf_stream))
                content = result.document.export_to_markdown()
                if len(content) > 5000:
                    ref.content = content

        # Instead try to download the article content directly from the HTML page
        if (not ref.content) and ref.html_url:
            html_request_response = requests.get(ref.html_url, timeout=30)

            if html_request_response.status_code == 200:
                doc = Document(html_request_response.text)
                article_html = doc.summary()
                content = md(article_html)
                if len(content) > 5000:
                    ref.content = content

    return ref

def fetch_reference_metadata_online(ref: Reference) -> Reference:
    """
    Enrich parsed references and return replacement list.
    """

    try:
        ref = fetch_crossref_metadata(ref)
        ref = fetch_openaccess_metadata(ref)
        ref = fetch_paper_content(ref)
    except Exception as e:
        print("Problem with reference:", ref.ref_id, "\n", type(e).__name__, e)
    return ref

enriched_references = list(map(fetch_reference_metadata_online, references))

An unexpected error occurred while opening the document paper.pdf
Traceback (most recent call last):
  File "/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 177, in __init__
    self._init_doc(backend, path_or_stream)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 221, in _init_doc
    self._backend = backend(self, path_or_stream=path_or_stream)
                    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/backend/docling_parse_backend.py", line 215, in __init__
    self._pdoc = pdfium.PdfDocument(self.path_or_stream, password=password)
                 ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/CitationChecker/citation_

Problem with reference: 68 
 ConversionError Input document paper.pdf is not valid.


An unexpected error occurred while opening the document paper.pdf
Traceback (most recent call last):
  File "/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 177, in __init__
    self._init_doc(backend, path_or_stream)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 221, in _init_doc
    self._backend = backend(self, path_or_stream=path_or_stream)
                    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/backend/docling_parse_backend.py", line 215, in __init__
    self._pdoc = pdfium.PdfDocument(self.path_or_stream, password=password)
                 ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/CitationChecker/citation_

Problem with reference: 85 
 ConversionError Input document paper.pdf is not valid.


An unexpected error occurred while opening the document paper.pdf
Traceback (most recent call last):
  File "/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 177, in __init__
    self._init_doc(backend, path_or_stream)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/datamodel/document.py", line 221, in _init_doc
    self._backend = backend(self, path_or_stream=path_or_stream)
                    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/CitationChecker/citation_checker_env/lib/python3.13/site-packages/docling/backend/docling_parse_backend.py", line 215, in __init__
    self._pdoc = pdfium.PdfDocument(self.path_or_stream, password=password)
                 ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srunser/Documents/CitationChecker/citation_

Problem with reference: 87 
 ConversionError Input document paper.pdf is not valid.


In [17]:
enriched_references

[Reference(ref_id=1, journal='PLoS Biology', title='An equatorial contractile mechanism drives cell elongation but not cell division.', authors=['Ivonne M. Sehring', 'Bo Dong', 'Elsa Denker', 'Punit Bhattachan', 'Wei Deng', 'Birthe T. Mathiesen', 'Di Jiang'], publication_year=2014, doi='10.1371/journal.pbio.1001781', html_url='https://doi.org/10.1371/journal.pbio.1001781', pdf_url='https://journals.plos.org/plosbiology/article/file?id=10.1371/journal.pbio.1001781&type=printable', is_open_access=True, content="<!-- image -->\n\n<!-- image -->\n\n## An Equatorial Contractile Mechanism Drives Cell Elongation but not Cell Division\n\nIvonne M. Sehring . , Bo Dong . ¤ , Elsa Denker, Punit Bhattachan, Wei Deng, Birthe T. Mathiesen, Di Jiang *\n\nSars International Centre for Marine Molecular Biology, University of Bergen, Bergen, Norway\n\n## Abstract\n\nCell shape changes and proliferation are two fundamental strategies for morphogenesis in animal development. During embryogenesis of the si

# Statements Extraction

In [8]:
class Statement(BaseModel):
    claim : Annotated[str, "One sentence summary of the scientific claim being made in the statement."]
    chunk_id : Annotated[int, "The id of the chunk of text from which the statement was extracted."]
    citations : Annotated[List[int], "The citation associated with the statement."]
    verification_result : Annotated[Literal["True", "Partially Verified", "False", "Unverified"], "The statement verification status."] | None

# Break the document in chunks
class Chunk(BaseModel):
    chunk_id : int
    text : str

# The global state used to track the extraction of statements from the text
class StatementExtractionState(TypedDict):

    # Contains the document 
    chunk_lst : List[Chunk]

    # The statement in the proper format
    statements: Annotated[List[Statement], operator.add]


# Substate used for the analysis of each chunk
class ChunkAnalysisState(TypedDict):
    chunk_id : int
    chunk_lst : List[Chunk]


In [9]:

def fan_out_statement_extraction_node(state: StatementExtractionState):
    """Break the initial document in chunks to parallelize the statement extraction"""
    chunk_lst = state["chunk_lst"]
    max_chunks = min(50, len(chunk_lst))

    return [
        Send(
            "statement_extraction_node",
            {
                "chunk_id": i,
                "chunk_lst": chunk_lst,
            },
        )
        for i in range(max_chunks)
    ]


def statement_extraction_node(state: ChunkAnalysisState) -> StatementExtractionState:
    """Extract from each document chunk the statements made and their associated citations"""

    chunk_lst = state["chunk_lst"]

    # Get the ID and text of the chunk
    chunk_id = state["chunk_id"]
    chunk_text = chunk_lst[chunk_id].text

    # Get the text surrounding the chunk
    previous_chunk_text = chunk_lst[chunk_id - 1].text if chunk_id > 0 else ""
    next_chunk_text = chunk_lst[chunk_id + 1].text if chunk_id < len(chunk_lst) - 1 else ""
    surrounding_text = previous_chunk_text + chunk_text + next_chunk_text

    class StatementOutput(BaseModel):
        statement: Optional[Statement]

    prompt = f"""
        Given the following chunk of text:
        {chunk_text}

        The id of this chunk of text is {chunk_id}

        If this chunk of text contains a scientific statement supported by one or several citations,
        extract the statement and its associated citations. Here is the text surrounding the chunk
        to give you more context and help you understand the claim that is made:
        {surrounding_text}

        Rules:
        - if the chunk of text contains no citation, return nothing
        - if the chunk of text contains a scientific claim, write a one sentence summary of the claim
        - only include claims that have at least one citation number
        - preserve the exact citation numbers from the text
        - do not invent citations
        - do not include brackets or parentheses around citation numbers
        - no markdown
        - no numbering
    """

    result = llm_low_temp.with_structured_output(StatementOutput).invoke(prompt)
    if result.statement is None:
        return {"statements": []}

    return {"statements": [result.statement]}


statement_extraction_graph_builder = StateGraph(StatementExtractionState)
statement_extraction_graph_builder.add_node("statement_extraction_node", statement_extraction_node)

statement_extraction_graph_builder.add_conditional_edges(START, fan_out_statement_extraction_node)
statement_extraction_graph_builder.add_edge("statement_extraction_node", END)
statement_extraction_graph = statement_extraction_graph_builder.compile()


statement_extraction_input = {
    "chunk_lst" : [Chunk(chunk_id=i, text=sent.text) for i, sent in enumerate(nlp(markdown).sents)],
}

statement_extraction_res = statement_extraction_graph.invoke(statement_extraction_input)
print(statement_extraction_res)

{'chunk_lst': [Chunk(chunk_id=0, text='## Resource\n\nhttps://doi.org/10.1038/s43588-024-00620-9\n\n## SimuCell3D: three-dimensional simulation of tissue mechanics with cell polarization\n\nReceived: 4 April 2023\n\nAccepted: 8 March 2024\n\nPublished online: 9 April 2024\n\nCheck for updates\n\nSteve Runser 1,2 , Roman Vetter 1,2 &amp; Dagmar Iber 1,2\n\n'), Chunk(chunk_id=1, text='The three-dimensional (3D) organization of cells determines tissue function and integrity, and changes markedly in development and disease.'), Chunk(chunk_id=2, text='Cell-based simulations have long been used to define the underlying mechanical principles.'), Chunk(chunk_id=3, text='However, high computational costs have so far limited simulations to either simplified cell geometries or small tissue patches.'), Chunk(chunk_id=4, text='Here, we present SimuCell3D, an efficient open-source program to simulate large tissues in three dimensions with subcellular resolution, growth, proliferation, extracellular 

# Statements Verification

In [10]:
# 